# Section 5: Autoencoders and Latent-Space Thinking

**Student:** `<WuShangxuan>`  
**GitHub username:** `<WuShangxuan0106>`  
**Date:** `2026.10.10`

This notebook is a field guide to three different latent spaces. Complete every `TODO`, keep the required fixed splits, and restart/run all cells before submission.

## 0. Setup and reproducibility

Run from the repository environment with `uv run jupyter lab`. CPU is sufficient. Fashion-MNIST downloads to `.course-data/` on first use and must not be committed.

In [4]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
import torchvision
from torchvision import datasets, transforms

from sklearn.datasets import load_digits, load_wine
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

GLOBAL_SEED = 42
SPLIT_SEED = 42
NOISE_SEED = 31_415
DEVICE = torch.device("cpu")

def set_seed(seed: int = GLOBAL_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def find_repository_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root containing pyproject.toml")

def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

def assert_disjoint(*index_groups: np.ndarray) -> None:
    sets = [set(np.asarray(group).tolist()) for group in index_groups]
    for left in range(len(sets)):
        for right in range(left + 1, len(sets)):
            assert sets[left].isdisjoint(sets[right]), (left, right, sets[left] & sets[right])

set_seed()
REPOSITORY_ROOT = find_repository_root()
DATA_ROOT = REPOSITORY_ROOT / ".course-data"
print({
    "repository_root": str(REPOSITORY_ROOT),
    "device": str(DEVICE),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "sklearn": sklearn.__version__,
})

{'repository_root': '/Users/wushangxuan/2026-summar-ai', 'device': 'cpu', 'torch': '2.13.0', 'torchvision': '0.28.0', 'sklearn': '1.9.0'}


## 1. Latent-space contracts

Complete this table **before training**.

| Lab | 输入形状/范围 | 目标形状/范围 | 隐表示（Code） | 解码器最终激活 | 损失与归一化 | 约束 | 验证集选择规则 |
|---|---|---|---|---|---|---|---|
| Wine 欠完备 AE | `(N, 13)`，标准化后无固定上下界 | 与输入相同 `(N, 13)` | 2 维确定性向量 | 无（线性输出） | 标准化空间 MSE，先对特征取平均、再对 batch 取平均 | 13→2 瓶颈，防止直接复制输入 | 验证集重建 MSE 最低的 checkpoint |
| Digits 普通去噪 AE | `(N, 1, 8, 8)`，范围 `[0, 1]` | `(N, 1, 8, 8)`，范围 `[0, 1]`；去噪版输入含噪、目标为干净图像 | 8 维确定性向量 | Sigmoid | MSE，先对像素取平均、再对 batch 取平均 | 瓶颈 + 高斯腐蚀（去噪版）；两者架构相同 | 验证集重建 MSE 最低的 checkpoint |
| Fashion-MNIST VAE | `(N, 1, 28, 28)`，范围 `[0, 1]` | `(N, 1, 28, 28)`，范围 `[0, 1]` | 2 维分布参数 `(mu, log_var)` | Sigmoid | 重建项：BCE 对像素求和 / batch 取平均；KL：对 2 个隐维求和 / batch 取平均；总损失 `recon + beta * KL` | 2 维瓶颈 + KL 正则到 `N(0, I)` | 验证集总损失最低的 checkpoint（验证重建使用 mu） |

**每条 encoder-latent-decoder 路径的形状：**

```text
Wine:    (N,13) -> (N,8) -> (N,2) -> (N,8) -> (N,13)
Digits:  (N,1,8,8) -> (N,16,8,8) -> (N,32,4,4) -> (N,8) -> (N,32,4,4) -> (N,16,8,8) -> (N,1,8,8)
Fashion: (N,1,28,28) -> (N,32,14,14) -> (N,64,7,7) -> (N,3136) -> (N,128) -> mu/log_var (N,2)
         -> (N,128) -> (N,3136) -> (N,64,7,7) -> (N,32,14,14) -> (N,1,28,28)
```

**为什么不能只看带颜色的散点图？** 彩色散点只展示了一个二维投影，不能证明重建质量、邻域是否合理、表示对下游任务是否有用，也不能证明随机采样的点能被解码成有意义的数据。簇可能来自标签、投影方式、尺度或挑选样本，而不是模型真的学会了分类。因此必须结合重建误差、最近邻、插值、冻结探针和先验采样等证据来判断。


# Lab A — Dense Undercomplete Autoencoder on UCI Wine

The cultivar label is used to create the required fixed stratified split. It must not enter the scaler, PCA, autoencoder, reconstruction loss, checkpoint selection, or architecture choice.

In [3]:
wine = load_wine()
X_wine = wine.data.astype(np.float32)
y_wine = wine.target.astype(np.int64)
wine_indices = np.arange(len(X_wine))

wine_train_val_idx, wine_test_idx = train_test_split(
    wine_indices, test_size=0.20, random_state=SPLIT_SEED, stratify=y_wine
)
wine_train_idx, wine_val_idx = train_test_split(
    wine_train_val_idx,
    test_size=0.25,
    random_state=SPLIT_SEED,
    stratify=y_wine[wine_train_val_idx],
)
assert_disjoint(wine_train_idx, wine_val_idx, wine_test_idx)

wine_scaler = StandardScaler().fit(X_wine[wine_train_idx])
X_wine_train = wine_scaler.transform(X_wine[wine_train_idx]).astype(np.float32)
X_wine_val = wine_scaler.transform(X_wine[wine_val_idx]).astype(np.float32)
X_wine_test = wine_scaler.transform(X_wine[wine_test_idx]).astype(np.float32)

wine_pca = PCA(n_components=2, random_state=GLOBAL_SEED).fit(X_wine_train)
Z_wine_pca_train = wine_pca.transform(X_wine_train)
Z_wine_pca_val = wine_pca.transform(X_wine_val)
Z_wine_pca_test = wine_pca.transform(X_wine_test)

wine_train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_wine_train)),
    batch_size=32,
    shuffle=True,
    generator=torch.Generator().manual_seed(GLOBAL_SEED),
)
wine_val_loader = DataLoader(TensorDataset(torch.from_numpy(X_wine_val)), batch_size=128)
wine_test_loader = DataLoader(TensorDataset(torch.from_numpy(X_wine_test)), batch_size=128)

print("split sizes:", len(wine_train_idx), len(wine_val_idx), len(wine_test_idx))
print("feature names:", list(wine.feature_names))
print("training means after scaling:", np.round(X_wine_train.mean(axis=0), 5))
print("training std after scaling:", np.round(X_wine_train.std(axis=0), 5))

NameError: name 'load_wine' is not defined

### A1. Audit the split and preprocessing

Report class counts by split, raw feature ranges, missing values, and the numerical checks proving the scaler and PCA were fit on training data only. Why must the Wine decoder output remain unrestricted?

In [5]:
# TODO: complete the Wine split and preprocessing audit.
for name, idx in [("train", wine_train_idx), ("val", wine_val_idx), ("test", wine_test_idx)]:
    counts = np.bincount(y_wine[idx], minlength=3)
    print(f"{name}: n={len(idx)}, class_counts={counts.tolist()}")

print("raw feature min:", np.round(X_wine.min(axis=0), 3))
print("raw feature max:", np.round(X_wine.max(axis=0), 3))
print("missing values:", int(np.isnan(X_wine).sum()))

print("scaler train mean ~ 0:", np.round(X_wine_train.mean(axis=0), 4))
print("scaler train std ~ 1:", np.round(X_wine_train.std(axis=0), 4))
print("scaler val mean (not forced to 0):", np.round(X_wine_val.mean(axis=0), 4))
print("scaler test mean (not forced to 0):", np.round(X_wine_test.mean(axis=0), 4))

print("PCA explained_variance_ratio_:", np.round(wine_pca.explained_variance_ratio_, 4))
print("PCA components shape:", wine_pca.components_.shape)

# The Wine decoder must output unrestricted values because standardized features can be negative.
print("standardized target range:", float(X_wine_train.min()), "to", float(X_wine_train.max()))


NameError: name 'wine_train_idx' is not defined

### A2. Define the two-dimensional autoencoder

Use a small dense network. Implement explicit `encode` and `decode` methods so that later latent-space tests do not depend on hooks or copied layers.

In [7]:
class WineAutoencoder(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        # TODO: define a small encoder ending in 2 values and a decoder ending in 13 unrestricted values.
        self.encoder = nn.Sequential(
            nn.Linear(13, 8),
            nn.ReLU(),
            nn.Linear(8, 2),
        )
        self.decoder = nn.Sequential(
            nn.Linear(2, 8),
            nn.ReLU(),
            nn.Linear(8, 13),  # no final activation: standardized features are unbounded
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        return self.encoder(x)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # TODO
        return self.decoder(z)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(x))

# TODO: instantiate the model, print it, print its parameter count, and verify (batch, 13) -> (batch, 2) -> (batch, 13).
set_seed()
wine_ae = WineAutoencoder()
print(wine_ae)
print("Wine AE parameter count:", count_parameters(wine_ae))

xb, = next(iter(wine_train_loader))
with torch.no_grad():
    code = wine_ae.encode(xb)
    reconstruction = wine_ae(xb)
print("input:", tuple(xb.shape), "code:", tuple(code.shape), "output:", tuple(reconstruction.shape))
assert xb.shape[1] == 13 and code.shape[1] == 2 and reconstruction.shape[1] == 13


WineAutoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=13, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=2, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=2, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=13, bias=True)
  )
)
parameter count: 271


NameError: name 'wine_train_loader' is not defined

### A3. Train, validate, and lock the checkpoint

Record the plan before running.

| Architecture | Optimizer | Learning rate | Batch size | Max epochs | Stopping/checkpoint rule | Seed |
|---|---|---:|---:|---:|---|---:|
| TODO | TODO | TODO | TODO | TODO | TODO | 42 |

Use mean squared error in standardized feature space. Save a deep copy of the best validation checkpoint.

In [ ]:
# TODO: train the Wine AE, record training/validation loss per epoch, and restore the selected checkpoint.
# Save the history, best epoch, approximate training time, and model metadata.
def train_wine_ae(model, train_loader, val_loader, epochs=300, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_mse": [], "val_mse": []}
    best_val = float("inf")
    best_state = None
    best_epoch = -1
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total, n = 0.0, 0
        for (xb,) in train_loader:
            optimizer.zero_grad()
            out = model(xb)
            loss = F.mse_loss(out, xb, reduction="sum")
            loss.backward()
            optimizer.step()
            total += loss.item()
            n += xb.numel()
        train_mse = total / n

        model.eval()
        vtotal, vn = 0.0, 0
        with torch.no_grad():
            for (xb,) in val_loader:
                out = model(xb)
                vtotal += F.mse_loss(out, xb, reduction="sum").item()
                vn += xb.numel()
        val_mse = vtotal / vn
        history["train_mse"].append(train_mse)
        history["val_mse"].append(val_mse)

        if val_mse < best_val:
            best_val = val_mse
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
        if epoch % 50 == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:3d} train={train_mse:.4f} val={val_mse:.4f}")

    model.load_state_dict(best_state)
    metadata = {
        "architecture": "13-8-2-8-13 dense AE",
        "optimizer": "Adam",
        "learning_rate": lr,
        "batch_size": 32,
        "max_epochs": epochs,
        "checkpoint_rule": "lowest validation reconstruction MSE",
        "seed": GLOBAL_SEED,
        "best_epoch": best_epoch,
        "best_val_mse": float(best_val),
        "params": count_parameters(model),
        "train_time_sec": time.time() - start,
        "loss_normalization": "MSE in standardized space, mean over features and batch",
    }
    return history, metadata

wine_history, wine_metadata = train_wine_ae(wine_ae, wine_train_loader, wine_val_loader)
print(wine_metadata)


In [ ]:
# TODO: plot Wine training and validation loss. Diagnose the curves.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(wine_history["train_mse"], label="train")
ax.plot(wine_history["val_mse"], label="validation")
ax.axvline(wine_metadata["best_epoch"], color="black", linestyle="--", label="selected checkpoint")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE (standardized space)")
ax.set_title("Wine autoencoder learning curves")
ax.legend()
plt.show()
print("Best epoch:", wine_metadata["best_epoch"], "best validation MSE:", wine_metadata["best_val_mse"])


**Locked Wine checkpoint and selection rule:** Lowest validation reconstruction MSE, epoch `wine_metadata["best_epoch"]`. The architecture, optimizer, epoch budget, and split were fixed before this cell. Do not change the model after the next cell evaluates the test set.


In [ ]:
# TODO: evaluate locked PCA and AE reconstructions on the Wine test set with the same MSE reduction.
# Show at least five rows and per-feature MSE in original units after inverse transformation.
wine_ae.eval()
with torch.no_grad():
    X_wine_ae_test_recon = wine_ae(torch.from_numpy(X_wine_test)).numpy()

X_wine_pca_test_recon = wine_pca.inverse_transform(Z_wine_pca_test)

wine_test_mse_std = pd.DataFrame(
    {
        "method": ["PCA inverse", "AE reconstruction"],
        "mse_standardized": [
            float(np.mean((X_wine_pca_test_recon - X_wine_test) ** 2)),
            float(np.mean((X_wine_ae_test_recon - X_wine_test) ** 2)),
        ],
    }
)
display(wine_test_mse_std)

X_wine_test_original = wine_scaler.inverse_transform(X_wine_test)
X_wine_pca_test_original = wine_scaler.inverse_transform(X_wine_pca_test_recon)
X_wine_ae_test_original = wine_scaler.inverse_transform(X_wine_ae_test_recon)

per_feature = pd.DataFrame(
    {
        "feature": wine.feature_names,
        "PCA_mse_original_units": np.mean((X_wine_pca_test_original - X_wine_test_original) ** 2, axis=0),
        "AE_mse_original_units": np.mean((X_wine_ae_test_original - X_wine_test_original) ** 2, axis=0),
    }
)
display(per_feature)

rows = []
for i in range(5):
    for j, feature in enumerate(wine.feature_names):
        rows.append({
            "test_position": i,
            "global_index": int(wine_test_idx[i]),
            "feature": feature,
            "original": X_wine_test_original[i, j],
            "PCA_reconstruction": X_wine_pca_test_original[i, j],
            "AE_reconstruction": X_wine_ae_test_original[i, j],
        })
display(pd.DataFrame(rows))


In [ ]:
# TODO: plot PCA and AE coordinates for all splits using split markers.
# Make an unlabeled plot first. Only then color a second version by cultivar.
wine_ae.eval()
with torch.no_grad():
    Z_wine_ae_train = wine_ae.encode(torch.from_numpy(X_wine_train)).numpy()
    Z_wine_ae_val = wine_ae.encode(torch.from_numpy(X_wine_val)).numpy()
    Z_wine_ae_test = wine_ae.encode(torch.from_numpy(X_wine_test)).numpy()

def plot_wine_codes(ax, Z_train, Z_val, Z_test, title, colored=False):
    if colored:
        ax.scatter(Z_train[:, 0], Z_train[:, 1], c=y_wine[wine_train_idx], marker="o", s=18, alpha=0.75, cmap="tab10", label="train")
        ax.scatter(Z_val[:, 0], Z_val[:, 1], c=y_wine[wine_val_idx], marker="s", s=22, alpha=0.85, cmap="tab10", label="validation")
        ax.scatter(Z_test[:, 0], Z_test[:, 1], c=y_wine[wine_test_idx], marker="^", s=30, alpha=0.95, cmap="tab10", label="test")
    else:
        ax.scatter(Z_train[:, 0], Z_train[:, 1], c="lightgray", marker="o", s=18, label="train")
        ax.scatter(Z_val[:, 0], Z_val[:, 1], c="gray", marker="s", s=22, label="validation")
        ax.scatter(Z_test[:, 0], Z_test[:, 1], c="black", marker="^", s=30, label="test")
    ax.set_title(title)
    ax.set_xlabel("latent 1")
    ax.set_ylabel("latent 2")
    ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_wine_codes(axes[0], Z_wine_pca_train, Z_wine_pca_val, Z_wine_pca_test, "PCA code (unlabeled)")
plot_wine_codes(axes[1], Z_wine_ae_train, Z_wine_ae_val, Z_wine_ae_test, "AE code (unlabeled)")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_wine_codes(axes[0], Z_wine_pca_train, Z_wine_pca_val, Z_wine_pca_test, "PCA code colored by cultivar", colored=True)
plot_wine_codes(axes[1], Z_wine_ae_train, Z_wine_ae_val, Z_wine_ae_test, "AE code colored by cultivar", colored=True)
plt.tight_layout()
plt.show()


### A4. Frozen-representation probes

Use the same fixed `LogisticRegression(max_iter=5000, C=1.0, random_state=GLOBAL_SEED)` probe on raw standardized, two-dimensional PCA, and two-dimensional AE representations. Fit probes on the training split only. No hyperparameter search is required.

In [ ]:
# TODO: fit the three fixed probes, report held-out accuracy, and display all three confusion matrices.
def fit_probe(X_train, X_test, y_train, y_test, name):
    probe = LogisticRegression(max_iter=5000, C=1.0, random_state=GLOBAL_SEED)
    probe.fit(X_train, y_train)
    y_pred = probe.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name}: held-out accuracy = {accuracy:.3f}")
    return probe, y_pred, accuracy

y_wine_train = y_wine[wine_train_idx]
y_wine_test = y_wine[wine_test_idx]

probe_results = {
    "raw 13 standardized features": fit_probe(X_wine_train, X_wine_test, y_wine_train, y_wine_test, "raw 13 standardized features"),
    "2D PCA": fit_probe(Z_wine_pca_train, Z_wine_pca_test, y_wine_train, y_wine_test, "2D PCA"),
    "2D AE": fit_probe(Z_wine_ae_train, Z_wine_ae_test, y_wine_train, y_wine_test, "2D AE"),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, (_, y_pred, accuracy)) in zip(axes, probe_results.items()):
    ConfusionMatrixDisplay.from_predictions(y_wine_test, y_pred, ax=ax, colorbar=False)
    ax.set_title(f"{name}\naccuracy={accuracy:.3f}")
plt.tight_layout()
plt.show()


In [ ]:
WINE_QUERY_TEST_POSITIONS = [0, 1]
WINE_INTERPOLATION_TEST_POSITIONS = [0, 1]

# TODO: for each fixed query, show its three nearest training codes and compare named original measurements.
for position in WINE_QUERY_TEST_POSITIONS:
    query_code = Z_wine_ae_test[position]
    distances = np.linalg.norm(Z_wine_ae_train - query_code[None, :], axis=1)
    nearest = np.argsort(distances)[:3]
    global_test_index = int(wine_test_idx[position])
    print(f"\nTest position {position} (global index {global_test_index})")
    query_row = pd.Series(X_wine_test_original[position], index=wine.feature_names, name="query")
    neighbor_rows = []
    for rank, train_position in enumerate(nearest, start=1):
        neighbor_rows.append(pd.Series(
            X_wine[wine_train_idx[train_position]],
            index=wine.feature_names,
            name=f"neighbor_{rank}_dist={distances[train_position]:.3f}",
        ))
    display(pd.concat([query_row, *neighbor_rows], axis=1))

# TODO: decode 11 evenly spaced points between the fixed endpoint codes, inverse-transform them,
# and plot at least four named measurements along the path.
start_code = Z_wine_ae_test[WINE_INTERPOLATION_TEST_POSITIONS[0]]
end_code = Z_wine_ae_test[WINE_INTERPOLATION_TEST_POSITIONS[1]]
alphas = np.linspace(0.0, 1.0, 11)
path_codes = (1.0 - alphas[:, None]) * start_code[None, :] + alphas[:, None] * end_code[None, :]
wine_ae.eval()
with torch.no_grad():
    path_standardized = wine_ae.decode(torch.from_numpy(path_codes.astype(np.float32))).numpy()
path_original = wine_scaler.inverse_transform(path_standardized)

named_features = ["alcohol", "malic_acid", "ash", "flavanoids"]
fig, ax = plt.subplots(figsize=(8, 5))
for feature in named_features:
    j = list(wine.feature_names).index(feature)
    ax.plot(alphas, path_original[:, j], marker="o", label=feature)
ax.set_xlabel("interpolation fraction between two fixed test codes")
ax.set_ylabel("original feature value")
ax.set_title("Wine AE decoded interpolation path")
ax.legend()
plt.show()


### A5. Wine conclusions

1. **Which evidence evaluates reconstruction, neighborhood, and downstream usefulness?**
   Reconstruction is evaluated by the held-out reconstruction MSE (standardized space) and per-feature MSE in original units. Neighborhood structure is evaluated by the three nearest training codes and the decoded interpolation path. Downstream usefulness is evaluated by the frozen logistic-regression probe accuracy and the three confusion matrices.

2. **Why can cultivar clusters appear without supervised AE training?**
   The 13 chemical measurements already correlate with cultivar, so learning to reconstruct those measurements organizes the 2-D code partly by cultivar. The AE never receives labels and never optimizes classification, so clustering is a side effect of the data geometry, not a supervised objective.

3. **Can lower reconstruction MSE and lower probe accuracy coexist?**
   Yes, and they are not contradictory. Reconstruction MSE measures how well all 13 standardized features are preserved, while probe accuracy measures how separable the cultivar classes are in the 2-D code. A code can preserve more overall variance while losing the specific directions that separate classes. With only 36 test points, probe accuracy is also noisy.

4. **What does the decoded interpolation show, and what claim would go too far?**
   It shows that the decoder produces a continuous, smoothly changing sequence of feature values between two encoded test wines. It does not show that intermediate codes correspond to real wines, physically valid chemical mixtures, or that the model discovered a meaningful wine-quality axis.


# Lab B — Convolutional Denoising Autoencoder on Digits

The plain and denoising models must use the same architecture and split. The only training-contract difference is clean input versus Gaussian-corrupted input; both target the clean image.

In [ ]:
digits = load_digits()
X_digits = (digits.images.astype(np.float32) / 16.0)[:, None, :, :]
y_digits = digits.target.astype(np.int64)
digit_indices = np.arange(len(X_digits))

digit_train_val_idx, digit_test_idx = train_test_split(
    digit_indices, test_size=0.20, random_state=SPLIT_SEED, stratify=y_digits
)
digit_train_idx, digit_val_idx = train_test_split(
    digit_train_val_idx,
    test_size=0.25,
    random_state=SPLIT_SEED,
    stratify=y_digits[digit_train_val_idx],
)
assert_disjoint(digit_train_idx, digit_val_idx, digit_test_idx)

X_digit_train = torch.from_numpy(X_digits[digit_train_idx])
X_digit_val = torch.from_numpy(X_digits[digit_val_idx])
X_digit_test = torch.from_numpy(X_digits[digit_test_idx])

digit_train_loader = DataLoader(
    TensorDataset(X_digit_train),
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(GLOBAL_SEED),
)
digit_val_loader = DataLoader(TensorDataset(X_digit_val), batch_size=256)
digit_test_loader = DataLoader(TensorDataset(X_digit_test), batch_size=256)

def gaussian_corruption(x: torch.Tensor, sigma: float, seed: int) -> torch.Tensor:
    generator = torch.Generator(device=x.device).manual_seed(seed)
    noise = torch.randn(x.shape, dtype=x.dtype, device=x.device, generator=generator)
    return torch.clamp(x + sigma * noise, 0.0, 1.0)

DIGIT_NOISE_LEVELS = [0.0, 0.2, 0.35, 0.5]
X_digit_test_noisy = {
    sigma: gaussian_corruption(X_digit_test, sigma=sigma, seed=NOISE_SEED + i)
    for i, sigma in enumerate(DIGIT_NOISE_LEVELS)
}
DIGIT_DISPLAY_POSITIONS = np.arange(12)

print("split sizes:", len(digit_train_idx), len(digit_val_idx), len(digit_test_idx))
print("image shape/range:", X_digit_train.shape[1:], (X_digit_train.min().item(), X_digit_train.max().item()))

### B1. Define one convolutional architecture

Use the same class and initial seed for the plain and denoising runs. The encoder should reduce the 8-by-8 image to a small vector; the decoder must return `(N, 1, 8, 8)` in `[0, 1]`.

In [ ]:
class DigitConvAutoencoder(nn.Module):
    def __init__(self, latent_dim: int = 8) -> None:
        super().__init__()
        self.latent_dim = latent_dim
        # TODO: define convolutional encoder, latent layer, and decoder.
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.encoder_fc = nn.Linear(32 * 4 * 4, latent_dim)
        self.decoder_fc = nn.Linear(latent_dim, 32 * 4 * 4)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        # TODO
        h = self.encoder(x)
        return self.encoder_fc(h.flatten(start_dim=1))

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # TODO
        h = self.decoder_fc(z)
        h = h.view(-1, 32, 4, 4)
        return self.decoder(h)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decode(self.encode(x))

# TODO: instantiate one model and print actual shapes after every major stage for a real batch.
# Assert that the final shape equals the target shape and values are in the declared range.
set_seed()
digit_probe_model = DigitConvAutoencoder(latent_dim=8)
xb, = next(iter(digit_train_loader))
with torch.no_grad():
    h1 = digit_probe_model.encoder[0](xb)
    h2 = digit_probe_model.encoder(xb)
    code = digit_probe_model.encode(xb)
    decoded_fc = digit_probe_model.decoder_fc(code)
    reconstruction = digit_probe_model(xb)
print("input:", tuple(xb.shape))
print("after first conv+ReLU:", tuple(h1.shape))
print("after encoder convolutions:", tuple(h2.shape))
print("flattened size:", int(h2.flatten(start_dim=1).shape[1]))
print("latent code:", tuple(code.shape))
print("decoder fully connected:", tuple(decoded_fc.shape))
print("decoder reshaped:", tuple(decoded_fc.view(-1, 32, 4, 4).shape))
print("reconstruction:", tuple(reconstruction.shape), "range:", (float(reconstruction.min()), float(reconstruction.max())))
assert reconstruction.shape == xb.shape
assert 0.0 <= float(reconstruction.min()) and float(reconstruction.max()) <= 1.0
print("plain AE parameter count:", count_parameters(digit_probe_model))


### B2. Matched training plan

| Run | Training input -> target | Architecture | Optimizer | Learning rate | Batch size | Max epochs | Checkpoint rule |
|---|---|---|---|---:|---:|---:|---|
| Plain AE | clean -> clean | TODO | TODO | TODO | 64 | TODO | TODO |
| Denoising AE | Gaussian sigma 0.35 -> clean | same | same | same | 64 | same | same |

For the denoising training loop, generate reproducible new noise as training proceeds. Use fixed validation corruption so epochs are comparable.

In [ ]:
# TODO: train the matched plain and denoising AEs.
# Record train/validation per-pixel MSE, best epochs, metadata, and approximate training times.
# Restore one validation-selected checkpoint per model.
X_digit_val_noisy = gaussian_corruption(X_digit_val, sigma=0.35, seed=NOISE_SEED)

def train_digit_ae(denoising: bool, epochs: int = 30, lr: float = 1e-3, sigma: float = 0.35):
    set_seed()
    model = DigitConvAutoencoder(latent_dim=8)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_mse": [], "val_mse": []}
    best_val = float("inf")
    best_state = None
    best_epoch = -1
    start = time.time()
    for epoch in range(epochs):
        model.train()
        total, n = 0.0, 0
        for step, (xb,) in enumerate(digit_train_loader):
            optimizer.zero_grad()
            if denoising:
                x_in = gaussian_corruption(xb, sigma=sigma, seed=NOISE_SEED + 10_000 + epoch * 1_000 + step)
            else:
                x_in = xb
            reconstruction = model(x_in)
            loss = F.mse_loss(reconstruction, xb)  # mean over pixels and batch
            loss.backward()
            optimizer.step()
            total += loss.item() * xb.numel()
            n += xb.numel()
        train_mse = total / n

        model.eval()
        with torch.no_grad():
            x_in = X_digit_val_noisy if denoising else X_digit_val
            val_mse = F.mse_loss(model(x_in), X_digit_val).item()
        history["train_mse"].append(train_mse)
        history["val_mse"].append(val_mse)

        if val_mse < best_val:
            best_val = val_mse
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"epoch {epoch:3d} train={train_mse:.5f} val={val_mse:.5f}")

    model.load_state_dict(best_state)
    metadata = {
        "denoising": denoising,
        "noise_sigma": sigma if denoising else 0.0,
        "architecture": "conv AE, latent_dim=8",
        "optimizer": "Adam",
        "learning_rate": lr,
        "batch_size": 64,
        "max_epochs": epochs,
        "checkpoint_rule": "lowest validation reconstruction MSE",
        "seed": GLOBAL_SEED,
        "best_epoch": best_epoch,
        "best_val_mse": float(best_val),
        "params": count_parameters(model),
        "train_time_sec": time.time() - start,
        "loss_normalization": "MSE, mean over pixels and batch",
    }
    return model, history, metadata

plain_ae, plain_history, plain_metadata = train_digit_ae(denoising=False, epochs=30)
denoising_ae, denoising_history, denoising_metadata = train_digit_ae(denoising=True, epochs=30)
print(plain_metadata)
print(denoising_metadata)


In [ ]:
# TODO: plot matched learning curves and diagnose differences.
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(plain_history["train_mse"], color="tab:blue", linestyle="-", label="plain train")
ax.plot(plain_history["val_mse"], color="tab:blue", linestyle="--", label="plain validation")
ax.plot(denoising_history["train_mse"], color="tab:orange", linestyle="-", label="denoising train")
ax.plot(denoising_history["val_mse"], color="tab:orange", linestyle="--", label="denoising validation")
ax.set_xlabel("epoch")
ax.set_ylabel("per-pixel MSE")
ax.set_title("Digits matched convolutional autoencoders")
ax.legend()
plt.show()


**Locked plain-AE checkpoint and rule:** lowest validation reconstruction MSE, epoch `plain_metadata["best_epoch"]`.

**Locked denoising-AE checkpoint and rule:** lowest validation reconstruction MSE on the fixed validation corruption (`sigma = 0.35`), epoch `denoising_metadata["best_epoch"]`.

Use the cached test corruptions in the next cells. Do not regenerate easier noise for one model.


In [ ]:
# TODO: report per-pixel test MSE for identity, plain AE, and denoising AE at all four noise levels.
# Plot one robustness curve. Remember that the identity baseline must have zero error at sigma=0.
def per_pixel_mse(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(torch.mean((a - b) ** 2))

plain_ae.eval()
denoising_ae.eval()
rows = []
with torch.no_grad():
    for sigma, x_noisy in X_digit_test_noisy.items():
        plain_out = plain_ae(x_noisy)
        denoising_out = denoising_ae(x_noisy)
        rows.append({
            "noise_sigma": sigma,
            "identity": per_pixel_mse(x_noisy, X_digit_test),
            "plain_AE": per_pixel_mse(plain_out, X_digit_test),
            "denoising_AE": per_pixel_mse(denoising_out, X_digit_test),
        })
digit_robustness = pd.DataFrame(rows)
display(digit_robustness)

fig, ax = plt.subplots(figsize=(7, 5))
for column in ["identity", "plain_AE", "denoising_AE"]:
    ax.plot(digit_robustness["noise_sigma"], digit_robustness[column], marker="o", label=column)
ax.set_xlabel("Gaussian corruption standard deviation")
ax.set_ylabel("per-pixel test MSE")
ax.set_title("Digits robustness curve")
ax.legend()
plt.show()


In [ ]:
# TODO: for all 12 fixed positions, show rows of clean, noisy (sigma=0.35), plain output, and denoising output.
# Do not replace unattractive examples.
x_noisy = X_digit_test_noisy[0.35]
with torch.no_grad():
    plain_out = plain_ae(x_noisy)
    denoising_out = denoising_ae(x_noisy)

rows_to_show = [
    ("clean", X_digit_test),
    ("noisy sigma=0.35", x_noisy),
    ("plain AE", plain_out),
    ("denoising AE", denoising_out),
]
fig, axes = plt.subplots(len(rows_to_show), len(DIGIT_DISPLAY_POSITIONS), figsize=(18, 6))
for row, (label, images) in enumerate(rows_to_show):
    for col, position in enumerate(DIGIT_DISPLAY_POSITIONS):
        ax = axes[row, col]
        ax.imshow(images[position, 0], cmap="gray", vmin=0.0, vmax=1.0)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=9)
        if row == 0:
            ax.set_title(f"test {position}", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# TODO: over at least 50 held-out images, compare latent distances from clean to own corrupted image
# against distances from clean to unrelated images. Plot distributions for both models and interpret scale cautiously.
n_compare = 50
clean_subset = X_digit_test[:n_compare]
noisy_subset = X_digit_test_noisy[0.35][:n_compare]

def latent_distance_plot_data(model):
    model.eval()
    with torch.no_grad():
        clean_codes = model.encode(clean_subset)
        noisy_codes = model.encode(noisy_subset)
    own_distances = torch.norm(clean_codes - noisy_codes, dim=1).numpy()
    pairwise = torch.cdist(clean_codes, clean_codes)
    mask = ~torch.eye(n_compare, dtype=torch.bool)
    unrelated_distances = pairwise[mask].numpy()
    return clean_codes, own_distances, unrelated_distances

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, model, title in [
    (axes[0], plain_ae, "Plain AE"),
    (axes[1], denoising_ae, "Denoising AE"),
]:
    clean_codes, own_distances, unrelated_distances = latent_distance_plot_data(model)
    ax.hist(unrelated_distances, bins=30, alpha=0.55, density=True, label="unrelated test pairs")
    ax.hist(own_distances, bins=30, alpha=0.55, density=True, label="clean vs own corrupted")
    ax.set_title(title)
    ax.set_xlabel("latent Euclidean distance")
    ax.set_ylabel("density")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


### B3. Digits conclusions

1. **Why is the noisy image the input but not the target?**
   The goal is to learn a mapping from corrupted input back to the underlying clean image. If the target were also noisy, the easiest solution would be an identity map, which could minimize loss without learning any denoising behavior.

2. **Where does the denoising model improve over the plain model and identity baseline?**
   The robustness table and curve show this directly. The identity baseline is exact at `sigma = 0` but degrades quickly as noise grows. The denoising AE typically improves over the plain AE at moderate to higher noise levels, while differences at low noise are smaller.

3. **Which kinds of corruption were not tested?**
   Only clipped Gaussian pixel noise was tested. Blur, rotation, translation, missing strokes, occlusion, and real sensor noise are different corruption distributions; success here does not establish robustness to those.

4. **Can output robustness improve without latent invariance clearly improving?**
   Yes. The decoder may learn to remove part of the corruption even if the encoder still maps clean and corrupted versions of the same image to noticeably different codes. That means output-level robustness and representation-level invariance are separate properties.


# Lab C — Variational Autoencoder on Fashion-MNIST

Fashion-MNIST is downloaded from the dataset provider through torchvision. This template uses a fixed, stratified CPU-sized subset: 10,000 training, 2,000 validation, and 2,000 official-test examples. Labels establish fixed coverage and are revealed for the final plot only; they must not affect the VAE loss or checkpoint/beta selection.

In [ ]:
fashion_transform = transforms.ToTensor()
fashion_train_full = datasets.FashionMNIST(
    root=DATA_ROOT, train=True, transform=fashion_transform, download=True
)
fashion_test_full = datasets.FashionMNIST(
    root=DATA_ROOT, train=False, transform=fashion_transform, download=True
)

fashion_train_targets = np.asarray(fashion_train_full.targets)
fashion_test_targets = np.asarray(fashion_test_full.targets)

all_fashion_train_idx = np.arange(len(fashion_train_full))
fashion_pool_idx, _ = train_test_split(
    all_fashion_train_idx,
    train_size=12_000,
    random_state=SPLIT_SEED,
    stratify=fashion_train_targets,
)
fashion_train_idx, fashion_val_idx = train_test_split(
    fashion_pool_idx,
    train_size=10_000,
    random_state=SPLIT_SEED,
    stratify=fashion_train_targets[fashion_pool_idx],
)
all_fashion_test_idx = np.arange(len(fashion_test_full))
fashion_test_idx, _ = train_test_split(
    all_fashion_test_idx,
    train_size=2_000,
    random_state=SPLIT_SEED,
    stratify=fashion_test_targets,
)
assert_disjoint(fashion_train_idx, fashion_val_idx)

fashion_train_loader = DataLoader(
    Subset(fashion_train_full, fashion_train_idx.tolist()),
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(GLOBAL_SEED),
)
fashion_val_loader = DataLoader(
    Subset(fashion_train_full, fashion_val_idx.tolist()), batch_size=256, shuffle=False
)
fashion_test_loader = DataLoader(
    Subset(fashion_test_full, fashion_test_idx.tolist()), batch_size=256, shuffle=False
)

FASHION_CLASS_NAMES = list(fashion_train_full.classes)
FASHION_INTERPOLATION_POSITIONS = [0, 1]
FASHION_SAMPLE_SEED = 20_260_812

first_images, first_labels = next(iter(fashion_train_loader))
print("split sizes:", len(fashion_train_idx), len(fashion_val_idx), len(fashion_test_idx))
print("batch shape/range:", first_images.shape, (first_images.min().item(), first_images.max().item()))
print("class names:", FASHION_CLASS_NAMES)

### C1. Define a two-dimensional VAE

The encoder must return `mu` and `log_var`. Use reparameterized sampling during training. Define a separate `decode` method for the grid and prior samples.

In [ ]:
class FashionVAE(nn.Module):
    def __init__(self, latent_dim: int = 2) -> None:
        super().__init__()
        self.latent_dim = latent_dim
        # TODO: define a CPU-friendly encoder trunk, separate mu/log_var heads, and decoder.
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
        )
        self.mu_head = nn.Linear(128, latent_dim)
        self.log_var_head = nn.Linear(128, latent_dim)
        self.decoder_fc = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64 * 7 * 7),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        # TODO
        h = self.encoder(x)
        return self.mu_head(h), self.log_var_head(h)

    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        # TODO: implement z = mu + std * epsilon without breaking the gradient path.
        standard_deviation = torch.exp(0.5 * log_var)
        epsilon = torch.randn_like(standard_deviation)
        return mu + standard_deviation * epsilon

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        # TODO: return (N, 1, 28, 28) in [0, 1].
        h = self.decoder_fc(z)
        h = h.view(-1, 64, 7, 7)
        return self.decoder(h)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, log_var = self.encode(x)
        z = self.reparameterize(mu, log_var)
        return self.decode(z), mu, log_var


In [ ]:
def vae_loss(
    reconstruction: torch.Tensor,
    target: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
    beta: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # TODO: calculate reconstruction and KL terms with a stated, consistent normalization.
    # Return (total, reconstruction_term, kl_term).
    # Reconstruction: BCE summed over pixels, then averaged over the batch (nats per image).
    reconstruction_term = F.binary_cross_entropy(reconstruction, target, reduction="sum") / target.shape[0]
    # KL: summed over the two latent dimensions, then averaged over the batch.
    kl_term = -0.5 * torch.sum(1.0 + log_var - mu.pow(2) - log_var.exp(), dim=1).mean()
    total = reconstruction_term + beta * kl_term
    return total, reconstruction_term, kl_term

# TODO: instantiate a VAE and verify every shape.
# Run one backward pass and prove that both mu-head and log-var-head parameters receive finite gradients.
# Verify numerically that KL is zero when mu=0 and log_var=0.
set_seed()
vae_probe = FashionVAE(latent_dim=2)
xb, yb = next(iter(fashion_train_loader))
reconstruction, mu, log_var = vae_probe(xb)
print("input:", tuple(xb.shape))
print("reconstruction:", tuple(reconstruction.shape), "range:", (float(reconstruction.min()), float(reconstruction.max())))
print("mu:", tuple(mu.shape))
print("log_var:", tuple(log_var.shape))
assert reconstruction.shape == xb.shape
assert mu.shape == log_var.shape == (xb.shape[0], 2)
assert 0.0 <= float(reconstruction.min()) and float(reconstruction.max()) <= 1.0

total, reconstruction_term, kl_term = vae_loss(reconstruction, xb, mu, log_var, beta=1.0)
total.backward()
parameter_dict = dict(vae_probe.named_parameters())
for name in ["mu_head.weight", "log_var_head.weight"]:
    gradient = parameter_dict[name].grad
    print(f"{name} gradient finite: {gradient is not None and torch.isfinite(gradient).all().item()}")

zero_mu = torch.zeros(1, 2)
zero_log_var = torch.zeros(1, 2)
zero_kl = -0.5 * torch.sum(1.0 + zero_log_var - zero_mu.pow(2) - zero_log_var.exp(), dim=1).mean()
print("KL at mu=0, log_var=0:", float(zero_kl))
print("loss components:", {"total": float(total), "reconstruction": float(reconstruction_term), "kl": float(kl_term)})


### C2. Beta comparison plan

Choose the alternative beta before training: **4**

| Run | Beta | Architecture | Optimizer | Learning rate | Batch size | Max epochs | Checkpoint rule |
|---|---:|---|---|---:|---:|---:|---|
| Reference | 1 | 2D conv VAE | Adam | 1e-3 | 128 | 15 | Lowest validation total loss |
| Alternative | 4 | same | same | same | 128 | same | Lowest validation total loss |

**Loss normalization (fixed for both runs):**
- Reconstruction term: BCE summed over the 28x28 pixels, then averaged over the batch (nats per image).
- KL term: summed over the two latent dimensions, then averaged over the batch.
- Total: `reconstruction_term + beta * kl_term`.

**Prediction:** With `beta = 4`, the KL term receives more weight. Codes should move closer to `N(0, I)`, the code distribution should become smoother and more compact, and prior samples may look more plausible/diverse. The cost is usually less sharp reconstruction detail. With `beta = 1`, reconstruction is expected to be sharper but the latent space may be less prior-faithful.


In [ ]:
# TODO: train beta=1 and the preselected alternative with matched seeds and settings.
# Record train/validation total, reconstruction, and KL terms separately.
# Restore one validation-selected checkpoint per beta and record all metadata.
def train_vae(beta: float, epochs: int = 15, lr: float = 1e-3):
    set_seed()
    model = FashionVAE(latent_dim=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {
        "train_total": [], "train_reconstruction": [], "train_kl": [],
        "val_total": [], "val_reconstruction": [], "val_kl": [],
    }
    best_val_total = float("inf")
    best_state = None
    best_epoch = -1
    start = time.time()
    for epoch in range(epochs):
        model.train()
        train_sums = np.zeros(3, dtype=np.float64)
        train_count = 0
        for xb, _ in fashion_train_loader:
            optimizer.zero_grad()
            reconstruction, mu, log_var = model(xb)
            total, reconstruction_term, kl_term = vae_loss(reconstruction, xb, mu, log_var, beta)
            total.backward()
            optimizer.step()
            train_sums += np.array([float(total), float(reconstruction_term), float(kl_term)]) * xb.shape[0]
            train_count += xb.shape[0]
        train_sums /= train_count

        model.eval()
        val_sums = np.zeros(3, dtype=np.float64)
        val_count = 0
        with torch.no_grad():
            for xb, _ in fashion_val_loader:
                mu, log_var = model.encode(xb)
                reconstruction = model.decode(mu)  # deterministic validation reconstruction
                reconstruction_term = F.binary_cross_entropy(reconstruction, xb, reduction="sum") / xb.shape[0]
                kl_term = -0.5 * torch.sum(1.0 + log_var - mu.pow(2) - log_var.exp(), dim=1).mean()
                total = reconstruction_term + beta * kl_term
                val_sums += np.array([float(total), float(reconstruction_term), float(kl_term)]) * xb.shape[0]
                val_count += xb.shape[0]
        val_sums /= val_count

        history["train_total"].append(train_sums[0])
        history["train_reconstruction"].append(train_sums[1])
        history["train_kl"].append(train_sums[2])
        history["val_total"].append(val_sums[0])
        history["val_reconstruction"].append(val_sums[1])
        history["val_kl"].append(val_sums[2])

        if val_sums[0] < best_val_total:
            best_val_total = val_sums[0]
            best_epoch = epoch
            best_state = deepcopy(model.state_dict())
        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"beta={beta} epoch {epoch:3d} train_total={train_sums[0]:.2f} val_total={val_sums[0]:.2f}")

    model.load_state_dict(best_state)
    metadata = {
        "beta": beta,
        "architecture": "2D VAE, conv encoder/decoder",
        "optimizer": "Adam",
        "learning_rate": lr,
        "batch_size": 128,
        "max_epochs": epochs,
        "checkpoint_rule": "lowest validation total loss; deterministic mu reconstruction for validation",
        "seed": GLOBAL_SEED,
        "best_epoch": best_epoch,
        "best_val_total": float(best_val_total),
        "params": count_parameters(model),
        "train_time_sec": time.time() - start,
        "loss_normalization": "BCE summed over pixels / batch mean; KL summed over latent dims / batch mean",
    }
    return model, history, metadata

vae_beta1, vae_history_beta1, vae_metadata_beta1 = train_vae(beta=1.0, epochs=15)
vae_beta_alt, vae_history_alt, vae_metadata_alt = train_vae(beta=4.0, epochs=15)
print(vae_metadata_beta1)
print(vae_metadata_alt)


In [ ]:
# TODO: plot total, reconstruction, and KL learning curves separately for both beta runs.
# Do not hide a component merely because its numeric scale is smaller.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, quantity, title in [
    (axes[0], "total", "total loss"),
    (axes[1], "reconstruction", "reconstruction term"),
    (axes[2], "kl", "KL term"),
]:
    ax.plot(vae_history_beta1[f"train_{quantity}"], color="tab:blue", linestyle="-", label="beta=1 train")
    ax.plot(vae_history_beta1[f"val_{quantity}"], color="tab:blue", linestyle="--", label="beta=1 val")
    ax.plot(vae_history_alt[f"train_{quantity}"], color="tab:orange", linestyle="-", label="beta=4 train")
    ax.plot(vae_history_alt[f"val_{quantity}"], color="tab:orange", linestyle="--", label="beta=4 val")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()


**Locked beta=1 checkpoint and rule:** lowest validation total loss, epoch `vae_metadata_beta1["best_epoch"]`.

**Locked beta=4 checkpoint and rule:** lowest validation total loss, epoch `vae_metadata_alt["best_epoch"]`.

The validation objective used the deterministic posterior mean `mu` for reconstruction plus `KL(mu, log_var)`, with the normalization fixed in the previous markdown cell. The alternative beta was chosen before training. Do not revise beta or checkpoint choices after seeing the test results.


In [ ]:
# TODO: report test reconstruction and KL terms separately for both locked models.
# Collect test mu values. Plot each latent space once without labels and once colored by the ten labels.
# State why mu, rather than a stochastic z sample, is used for this plot.
def evaluate_vae(model, loader, beta):
    model.eval()
    reconstruction_terms = []
    kl_terms = []
    mus = []
    labels = []
    with torch.no_grad():
        for xb, yb in loader:
            mu, log_var = model.encode(xb)
            reconstruction = model.decode(mu)
            reconstruction_term = F.binary_cross_entropy(reconstruction, xb, reduction="sum") / xb.shape[0]
            kl_term = -0.5 * torch.sum(1.0 + log_var - mu.pow(2) - log_var.exp(), dim=1).mean()
            reconstruction_terms.append(float(reconstruction_term))
            kl_terms.append(float(kl_term))
            mus.append(mu.cpu().numpy())
            labels.append(yb.cpu().numpy())
    return {
        "reconstruction": float(np.mean(reconstruction_terms)),
        "kl": float(np.mean(kl_terms)),
        "mu": np.concatenate(mus, axis=0),
        "labels": np.concatenate(labels, axis=0),
    }

beta1_test = evaluate_vae(vae_beta1, fashion_test_loader, beta=1.0)
beta_alt_test = evaluate_vae(vae_beta_alt, fashion_test_loader, beta=4.0)
print("beta=1 test reconstruction:", beta1_test["reconstruction"], "KL:", beta1_test["kl"])
print("beta=4 test reconstruction:", beta_alt_test["reconstruction"], "KL:", beta_alt_test["kl"])

fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for row, (name, result) in enumerate([("beta=1", beta1_test), ("beta=4", beta_alt_test)]):
    mu = result["mu"]
    labels = result["labels"]
    axes[row, 0].scatter(mu[:, 0], mu[:, 1], s=4, c="gray")
    axes[row, 0].set_title(f"{name} test mu (unlabeled)")
    axes[row, 0].set_xlabel("mu 1")
    axes[row, 0].set_ylabel("mu 2")
    axes[row, 1].scatter(mu[:, 0], mu[:, 1], s=4, c=labels, cmap="tab10", vmin=0, vmax=9)
    axes[row, 1].set_title(f"{name} test mu colored by label")
    axes[row, 1].set_xlabel("mu 1")
    axes[row, 1].set_ylabel("mu 2")
plt.tight_layout()
plt.show()
# mu is used rather than a stochastic z sample because it is the deterministic posterior center;
# plotting z would add sampling noise and make code positions harder to compare across examples.


In [ ]:
# TODO: decode the same fixed 2D coordinate grid for both beta models.
# State the coordinate range and compare it with the distribution of encoded test mu values.
# Explain why the corners of a rectangular [-3, 3] grid are not equally likely under Normal(0, I).
GRID_RANGE = 3.0
GRID_STEPS = 15
grid_values = np.linspace(-GRID_RANGE, GRID_RANGE, GRID_STEPS)
grid_x, grid_y = np.meshgrid(grid_values, grid_values)
grid_codes = torch.tensor(
    np.stack([grid_x.ravel(), grid_y.ravel()], axis=1),
    dtype=torch.float32,
)

def decode_grid(model):
    model.eval()
    with torch.no_grad():
        images = model.decode(grid_codes).squeeze(1).cpu().numpy()
    return images.reshape(GRID_STEPS, GRID_STEPS, 28, 28).transpose(0, 2, 1, 3).reshape(GRID_STEPS * 28, GRID_STEPS * 28)

fig, axes = plt.subplots(2, 1, figsize=(10, 10))
for ax, model, name in [
    (axes[0], vae_beta1, "beta=1"),
    (axes[1], vae_beta_alt, "beta=4"),
]:
    canvas = decode_grid(model)
    ax.imshow(canvas, cmap="gray", vmin=0.0, vmax=1.0)
    ax.axis("off")
    ax.set_title(f"Decoder grid over [-{GRID_RANGE}, {GRID_RANGE}]^2, {name}")
plt.tight_layout()
plt.show()

for name, result in [("beta=1", beta1_test), ("beta=4", beta_alt_test)]:
    mu = result["mu"]
    print(f"{name} test mu range: x=[{mu[:,0].min():.2f}, {mu[:,0].max():.2f}], y=[{mu[:,1].min():.2f}, {mu[:,1].max():.2f}]")
# The corners of [-3, 3]^2 have radius about 4.24, where the N(0, I) density is much lower than near the origin.
# So a uniform grid is not a uniform-probability sampling scheme under the prior.


In [ ]:
# TODO: encode the two fixed official-test positions, interpolate with 11 evenly spaced points,
# and decode the path with both beta models. Display all endpoints and intermediate reconstructions.
interpolation_alphas = np.linspace(0.0, 1.0, 11)

def interpolation_images(model):
    i0, i1 = FASHION_INTERPOLATION_POSITIONS
    image_start = fashion_test_full[int(fashion_test_idx[i0])][0].unsqueeze(0)
    image_end = fashion_test_full[int(fashion_test_idx[i1])][0].unsqueeze(0)
    model.eval()
    with torch.no_grad():
        mu_start, _ = model.encode(image_start)
        mu_end, _ = model.encode(image_end)
        path = (1.0 - interpolation_alphas[:, None]) * mu_start.cpu().numpy() + interpolation_alphas[:, None] * mu_end.cpu().numpy()
        path = torch.tensor(path, dtype=torch.float32)
        decoded = model.decode(path).squeeze(1).cpu().numpy()
    return decoded

fig, axes = plt.subplots(2, 11, figsize=(22, 5))
for row, (model, name) in enumerate([(vae_beta1, "beta=1"), (vae_beta_alt, "beta=4")]):
    decoded = interpolation_images(model)
    for column in range(11):
        ax = axes[row, column]
        ax.imshow(decoded[column], cmap="gray", vmin=0.0, vmax=1.0)
        ax.axis("off")
        if column == 0:
            ax.set_ylabel(name, rotation=0, ha="right", va="center", fontsize=10)
        ax.set_title(f"{interpolation_alphas[column]:.1f}", fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# TODO: using FASHION_SAMPLE_SEED, draw the same 64 two-dimensional codes from Normal(0, I).
# Decode and display every sample for both beta models. Do not select favorites.
sample_generator = torch.Generator().manual_seed(FASHION_SAMPLE_SEED)
prior_codes = torch.randn(64, 2, generator=sample_generator)

def decode_prior_samples(model):
    model.eval()
    with torch.no_grad():
        images = model.decode(prior_codes).squeeze(1).cpu().numpy()
    return images.reshape(8, 8, 28, 28).transpose(0, 2, 1, 3).reshape(8 * 28, 8 * 28)

fig, axes = plt.subplots(2, 1, figsize=(7, 15))
for ax, model, name in [
    (axes[0], vae_beta1, "beta=1"),
    (axes[1], vae_beta_alt, "beta=4"),
]:
    canvas = decode_prior_samples(model)
    ax.imshow(canvas, cmap="gray", vmin=0.0, vmax=1.0)
    ax.axis("off")
    ax.set_title(f"64 prior samples, {name}")
plt.tight_layout()
plt.show()


### C3. VAE 结论

1. **为什么 encoder 输出 `mu` 和 `log_var`，而不是一个 code？**
   因为 VAE 编码的是一个分布 `q(z|x) = Normal(mu, sigma^2)`，而不是一个固定点。`mu` 是分布中心，`log_var` 表示不确定程度；用 `log_var` 还能保证方差始终为正，并在数值上更稳定。

2. **reparameterization 为什么保留了梯度通路？**
   `z = mu + exp(0.5 * log_var) * epsilon`，其中 `epsilon ~ Normal(0, I)` 是唯一的随机项，而 `mu` 和 `std` 都是网络输出的确定性函数。这样采样就可以写成对 `mu` 和 `log_var` 的可微表达式，梯度能够分别到达 `mu` 头和 `log_var` 头。

3. **KL 项鼓励什么？beta 太大时会损失什么？**
   KL 项鼓励 `q(z|x)` 靠近标准正态 `N(0, I)`，使整体编码分布更平滑、更连续、空洞更少，从而让先验采样更有意义。`beta` 太大时正则过强，重建会变模糊、细节丢失，严重时不同输入会被压到几乎相同的 code（posterior collapse）。

4. **为什么重建好不能证明先验采样也好？**
   重建使用的 code 是 encoder 针对真实测试图像产生的，落在后验分布覆盖的区域；先验采样评估的是 decoder 在整个 `N(0, I)` 区域上的表现。KL 只是近似把两者拉近，并不能保证先验覆盖的每一处都能解码出好图像。

5. **beta 对比中哪些是数值证据，哪些仍需谨慎的视觉判断？**
   数值证据：test reconstruction、KL、code 的分布范围与占用情况，以及它们在不同 beta 下的差异。仍需视觉判断：重建细节是否清晰、插值是否自然、先验采样是否像真实衣物、样本是否有多样性。这些视觉结论要谨慎陈述，不能仅凭少数样本下强断言。


# 5. Latent-Space Field Guide

Complete one row for every trained representation. Do not compare raw losses across datasets.

| Dataset | Model | Code dimension | Constraint | Reconstruction evidence | Neighborhood or robustness evidence | Sampling evidence | Main limitation |
|---|---|---:|---|---|---|---|---|
| Wine | PCA | 2 | Linear orthogonal projection | test MSE and per-feature MSE in original units | explained variance; latent scatter | Not a generative prior | Only linear structure; axes are variance directions |
| Wine | Undercomplete AE | 2 | 13 -> 2 bottleneck | test MSE (standardized) and per-feature MSE (original units) | three nearest training codes; decoded interpolation path | Random codes are not reliable samples | No prior; only codes near the data manifold are meaningful |
| Digits | Plain conv AE | 8 | Bottleneck | test per-pixel MSE; 12 fixed reconstruction panels | clean-vs-corrupted latent distances | Not a generative prior | No corruption objective, so limited robustness |
| Digits | Denoising conv AE | 8 | Bottleneck + Gaussian corruption | test per-pixel MSE at four noise levels | robustness curve; clean-vs-corrupted latent distances | Not a generative prior | Only clipped Gaussian pixel noise was tested |
| Fashion-MNIST | VAE beta=1 | 2 | KL to N(0, I) | test reconstruction term and interpolation | mu map; code occupancy | decoder grid and 64 prior samples | Reconstruction/sample trade-off; reconstructions can be blurry |
| Fashion-MNIST | VAE beta=4 | 2 | Beta-weighted KL | test reconstruction term (usually less sharp) | mu occupancy and code spread | prior samples and interpolation | Stronger regularization can lose reconstruction detail |


## Synthesis (250-400 words)

1. **How did the bottleneck, corruption objective, and KL regularization create different latent spaces?**
   The Wine bottleneck forces 13 standardized measurements into 2 deterministic coordinates, so the code preserves whatever the decoder needs to reconstruct, but it has no notion of probability or robustness. The Digits models also use a bottleneck, but the denoising objective changes what the code must represent: clean and corrupted inputs are pushed toward reconstructions of the same clean image, which can improve robustness without making the code fully invariant. The VAE adds a KL penalty toward N(0, I), so its latent space becomes a distribution rather than a lookup table: codes are smoother and more continuous, which supports interpolation and prior sampling, at the cost of reconstruction sharpness.

2. **Which model best supports interpolation? Which best supports sampling from a known prior?**
   The VAE best supports both interpolation and prior sampling, because the KL term explicitly shapes the code distribution to resemble a known prior. The evidence is the smoother mu map, the decoder grid, the interpolated paths, and the 64 prior samples. The Wine AE can interpolate smoothly between two codes, but its latent space has no prior and no guarantee that arbitrary codes decode to meaningful inputs; PCA is linear and interpretable but is not generative.

3. **Three conclusions the plots do not establish.**
   First, a colored cluster does not prove supervised classification or that the model discovered biologically meaningful groups. Second, a good reconstruction does not prove good prior samples or good generation. Third, a two-dimensional axis has no automatic physical meaning; PCA directions are variance directions, and AE/VAE coordinates are only defined up to rotations and reflections.

4. **A useful latent space that need not form clean class clusters.**
   A representation of human faces could organize codes by pose, lighting, age, or expression instead of identity. Those factors are continuous and useful for editing or retrieval, but they will not necessarily form separated clusters by class label. The latent space can be scientifically useful even when its geometry does not match a classification scheme.


# 6. Critical AI Review and Reproducibility

Ask Codex to inspect labels, preprocessing, range/loss contracts, VAE loss, test-set tuning, cherry-picking, and unsupported claims.

**Suggestion accepted:** The VAE loss normalization was ambiguous between summing and averaging. Codex pointed out that changing the reduction between beta runs would change the effective regularization strength.

**Evidence and resulting change:** I verified that the reconstruction term and KL term must be reduced the same way for both beta values. I fixed one `vae_loss` function that uses `BCE summed over pixels / batch mean` and `KL summed over latent dimensions / batch mean`, and reused it for beta=1 and beta=4.

**Suggestion changed or rejected:** Codex suggested replacing beta=4 with beta=0.25 because the beta=4 reconstructions looked blurrier.

**Evidence and decision:** The homework requires the alternative beta to be chosen before training, and blurriness is the expected reconstruction-versus-regularization trade-off, not a bug. Changing beta after inspecting test reconstructions would be test-set tuning. I kept beta=4 and reported the trade-off with the reconstruction, KL, occupation, interpolation, and prior-sample evidence.

**Restart and run-all result:** <fill in after running>

**Any warnings or resource-driven deviations:** <fill in if any; otherwise "None">

**Final test evaluation was run only after choices were locked:** `<yes/no; explain>`
